# Bootstrap Robustness Check — EA CPD

**Purpose:** Same method. Given λ=1.02 (near-perfect) and perfect alpha-sweep
stability already confirmed, expecting strong bootstrap robustness, consistent
with AA CPD's pattern (MAF-floored smoker-subsample analyses outperforming
full-cohort smoking_status analyses on every check so far).

**Input:** `pc_input_cpd_final.npy`, `pc_col_names_cpd_final.json`.

In [1]:
import numpy as np
import json
import os
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import fisherz
from causallearn.utils.PCUtils.BackgroundKnowledge import BackgroundKnowledge
from causallearn.graph.GraphNode import GraphNode
from collections import defaultdict
import time

out_dir = r"C:\Users\user\Downloads\GSE148812_clean"
X_pc_full_cpd_ea = np.load(os.path.join(out_dir, "pc_input_cpd_final.npy"))
with open(os.path.join(out_dir, "pc_col_names_cpd_final.json")) as f:
    col_names_cpd_ea = json.load(f)

n_nodes = len(col_names_cpd_ea)
outcome_idx = col_names_cpd_ea.index("CPD")
n_samples = X_pc_full_cpd_ea.shape[0]

def get_direct_parents_ea_cpd(X, alpha_val=0.001):
    bk = BackgroundKnowledge()
    nodes = [GraphNode(name) for name in col_names_cpd_ea]
    for i in range(n_nodes - 1):
        bk.add_node_to_tier(nodes[i], 0)
    bk.add_node_to_tier(nodes[outcome_idx], 1)

    cg = pc(data=X, alpha=alpha_val, indep_test=fisherz, stable=True,
            uc_rule=0, uc_priority=2, background_knowledge=bk,
            verbose=False, show_progress=False, node_names=col_names_cpd_ea)

    adj = cg.G.graph
    direct_parents = set()
    for i in range(n_nodes):
        for j in range(i+1, n_nodes):
            if adj[i,j] == -1 and adj[j,i] == 1 and col_names_cpd_ea[j] == "CPD":
                direct_parents.add(col_names_cpd_ea[i])
            elif adj[i,j] == 1 and adj[j,i] == -1 and col_names_cpd_ea[i] == "CPD":
                direct_parents.add(col_names_cpd_ea[j])
            elif adj[i,j] == -1 and adj[j,i] == -1:
                if col_names_cpd_ea[i] == "CPD":
                    direct_parents.add(col_names_cpd_ea[j])
                elif col_names_cpd_ea[j] == "CPD":
                    direct_parents.add(col_names_cpd_ea[i])
    return direct_parents

n_bootstraps = 100
edge_counts_ea_cpd = defaultdict(int)
n_failed = 0
n_succeeded = 0

np.random.seed(0)
start = time.time()
for b in range(n_bootstraps):
    boot_idx = np.random.choice(n_samples, n_samples, replace=True)
    X_boot = X_pc_full_cpd_ea[boot_idx]
    try:
        parents = get_direct_parents_ea_cpd(X_boot)
        for gene in parents:
            edge_counts_ea_cpd[gene] += 1
        n_succeeded += 1
    except (ValueError, np.linalg.LinAlgError):
        n_failed += 1
        continue
    if (b + 1) % 20 == 0:
        print(f"Completed {b+1}/{n_bootstraps}, elapsed {time.time()-start:.1f}s")

print(f"\nTotal time: {time.time()-start:.1f}s")
print(f"Succeeded: {n_succeeded}, Failed: {n_failed}")
print(f"\nEdge stability across {n_succeeded} successful bootstraps:")
for gene, count in sorted(edge_counts_ea_cpd.items(), key=lambda x: -x[1]):
    pct = count / n_succeeded * 100
    print(f"  {gene}: {count}/{n_succeeded} ({pct:.0f}%)")

c:\Users\user\Desktop\ai causal\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Completed 20/100, elapsed 0.1s
Completed 40/100, elapsed 0.3s
Completed 60/100, elapsed 0.5s
Completed 80/100, elapsed 0.7s
Completed 100/100, elapsed 0.9s

Total time: 0.9s
Succeeded: 100, Failed: 0

Edge stability across 100 successful bootstraps:
  exm1614640-0_T_R_1919086982: 84/100 (84%)
  exm-rs12188164-131_B_R_1990478895: 72/100 (72%)
  exm1372139-0_T_F_1921594655: 61/100 (61%)
  exm2272793-0_T_R_1984856876: 52/100 (52%)
